# Use the RDF Data of the Swiss Political Parties

In [1]:
# load graph from the Turtle file
from rdflib import Graph

g = Graph()
g.parse("../data/output/parties.ttl", format="turtle")

<Graph identifier=N5251363c15b44d34818b334a0afe876a (<class 'rdflib.graph.Graph'>)>

In [2]:
import pandas as pd

def sparql_to_df(graph: Graph, query: str) -> pd.DataFrame:
    result = graph.query(query)
    return pd.DataFrame(
        [
            [val.toPython() if val is not None else None for val in row]
            for row in result
        ],
        columns=[str(var) for var in result.vars]
    )

## Number of Versions per Party

In [3]:
query = """

PREFIX vl: <https://version.link/>

SELECT ?party (COUNT(?version) AS ?count) WHERE {
    ?party a vl:Identity .
    ?version vl:identity ?party .
} GROUP BY ?party ORDER BY DESC(?count)

"""

In [4]:
df = sparql_to_df(g, query)

display(df)

,party,count
0,https://politics.ld.admin.ch/party-id/128,11
1,https://politics.ld.admin.ch/party-id/311,9
2,https://politics.ld.admin.ch/party-id/35,8
3,https://politics.ld.admin.ch/party-id/37,8
4,https://politics.ld.admin.ch/party-id/103,7
...,...,...
415,https://politics.ld.admin.ch/party-id/8,1
416,https://politics.ld.admin.ch/party-id/11,1
417,https://politics.ld.admin.ch/party-id/6,1
418,https://politics.ld.admin.ch/party-id/3,1


## Identities without Versions

Should not happen

In [5]:
query = """

PREFIX vl: <https://version.link/>

SELECT ?id WHERE {
    ?id a vl:Identity .
    FILTER NOT EXISTS {
        ?id vl:version ?version .
    }
}
    
"""

In [6]:
df = sparql_to_df(g, query)

display(df)

,id


## Parties without Names

In [7]:
query = """

PREFIX vl: <https://version.link/>
PREFIX schema: <http://schema.org/>

SELECT ?id WHERE {
    ?id a vl:Identity .
    FILTER NOT EXISTS {
        ?id schema:name ?name .
    }
}
    
"""

In [8]:
df = sparql_to_df(g, query)

display(df)

,id
0,https://politics.ld.admin.ch/party-id/353
1,https://politics.ld.admin.ch/party-id/135
2,https://politics.ld.admin.ch/party-id/158
3,https://politics.ld.admin.ch/party-id/194
4,https://politics.ld.admin.ch/party-id/226
5,https://politics.ld.admin.ch/party-id/250
6,https://politics.ld.admin.ch/party-id/258
7,https://politics.ld.admin.ch/party-id/260
8,https://politics.ld.admin.ch/party-id/262
9,https://politics.ld.admin.ch/party-id/304


## All Names of a Party

Caution: This query will return the names of all versions. If the name is the same for multiple versions, it will be returned multiple times.

In [16]:
query = """

PREFIX vl: <https://version.link/>
PREFIX schema: <http://schema.org/>

SELECT ?name ?validFrom ?validThrough WHERE {
    ?version vl:identity <https://politics.ld.admin.ch/party-id/5> ;
        schema:name ?name ;
        schema:validFrom ?validFrom .

    OPTIONAL {
        ?version schema:validThrough ?validThrough .
    }
} ORDER BY ?validFrom
    
"""

In [18]:
df = sparql_to_df(g, query)

display(df)

,name,validFrom,validThrough
0,Föderation der grünen Parteien der Schweiz,1983-05-28,1986-06-30
1,Fédération des partis écologistes de Suisse,1983-05-28,1986-06-30
2,Grüne Partei der Schweiz,1986-07-01,1993-09-04
3,Parti écologiste suisse,1986-07-01,1993-09-04
4,Grüne – Grüne Partei der Schweiz,1993-09-05,2007-07-18
5,Les Verts - Parti écologiste suisse,1993-09-05,2007-07-18
6,Grüne – Grüne Partei der Schweiz,2007-07-19,2018-06-30
7,Les Verts - Parti écologiste suisse,2007-07-19,2018-06-30
8,Grüne – Grüne Partei der Schweiz,2018-07-01,2021-03-26
9,Les Verts - Parti écologiste suisse,2018-07-01,2021-03-26


## All Relationships of a Party Version

In [11]:
query = """

PREFIX vl: <https://version.link/>
PREFIX schema: <http://schema.org/>

SELECT ?version ?from ?name WHERE {
    <https://politics.ld.admin.ch/party-version/1_1971-09-22> (vl:successor|vl:predecessor)+ ?version .

    ?version schema:validFrom ?from .

    OPTIONAL {
        ?version schema:name ?name .
        FILTER (lang(?name) = "de")
    }

} ORDER BY ?from
    
"""

In [12]:
df = sparql_to_df(g, query)

display(df)

,version,from,name
0,https://politics.ld.admin.ch/party-version/22_...,1840-11-05,Vorläufer der CVP
1,https://politics.ld.admin.ch/party-version/16_...,1880-01-01,Vorläufer der Demokraten
2,https://politics.ld.admin.ch/party-version/17_...,1905-03-19,Schweizerische Demokratische Partei
3,https://politics.ld.admin.ch/party-version/10_...,1912-04-22,Schweizerische Konservative Volkspartei
4,https://politics.ld.admin.ch/party-version/209...,1920-01-01,"Vorläufer der Bauern-, Gewerbe und Bürgerpartei"
5,https://politics.ld.admin.ch/party-version/11_...,1936-12-23,"Bauern-, Gewerbe- und Bürgerpartei"
6,https://politics.ld.admin.ch/party-version/18_...,1941-07-01,Demokratische Partei der Schweiz
7,https://politics.ld.admin.ch/party-version/10_...,1957-07-01,Konservativ-Christlichsoziale Volkspartei
8,https://politics.ld.admin.ch/party-version/10_...,1970-07-01,NaN
9,https://politics.ld.admin.ch/party-version/10_...,1970-12-12,Christlichdemokratische Volkspartei der Schweiz


## Visualization of the Relationships of a Party Version

In [13]:
query = """

PREFIX vl: <https://version.link/>
PREFIX schema: <http://schema.org/>

SELECT ?source ?target ?source_name ?target_name WHERE {
    
    <https://politics.ld.admin.ch/party-version/1_1971-09-22> (vl:successor|vl:predecessor)+ ?source .

    ?source vl:successor ?target .
        
    OPTIONAL {
        ?source schema:name ?source_name .
        FILTER (lang(?source_name) = "de")
    }
    OPTIONAL {
        ?target schema:name ?target_name .
        FILTER (lang(?target_name) = "de")
    }
}
    
"""

In [14]:
df = sparql_to_df(g, query)

display(df)

,source,target,source_name,target_name
0,https://politics.ld.admin.ch/party-version/13_...,https://politics.ld.admin.ch/party-version/4_2...,Bürgerlich-Demokratische Partei Schweiz,Die Mitte Schweiz
1,https://politics.ld.admin.ch/party-version/10_...,https://politics.ld.admin.ch/party-version/4_2...,Christlichdemokratische Volkspartei der Schweiz,Die Mitte Schweiz
2,https://politics.ld.admin.ch/party-version/10_...,https://politics.ld.admin.ch/party-version/10_...,Christlichdemokratische Volkspartei der Schweiz,Christlichdemokratische Volkspartei der Schweiz
3,https://politics.ld.admin.ch/party-version/10_...,https://politics.ld.admin.ch/party-version/66_...,Christlichdemokratische Volkspartei der Schweiz,Christlich-Soziale Partei der Schweiz
4,https://politics.ld.admin.ch/party-version/66_...,https://politics.ld.admin.ch/party-version/66_...,Christlich-Soziale Partei der Schweiz,Mitte Links – Christlich-soziale Partei Schweiz
5,https://politics.ld.admin.ch/party-version/10_...,https://politics.ld.admin.ch/party-version/10_...,Christlichdemokratische Volkspartei der Schweiz,Christlichdemokratische Volkspartei der Schweiz
6,https://politics.ld.admin.ch/party-version/10_...,https://politics.ld.admin.ch/party-version/65_...,Christlichdemokratische Volkspartei der Schweiz,Christlichsoziale Partei der Schweiz
7,https://politics.ld.admin.ch/party-version/65_...,https://politics.ld.admin.ch/party-version/65_...,Christlichsoziale Partei der Schweiz,NaN
8,https://politics.ld.admin.ch/party-version/10_...,https://politics.ld.admin.ch/party-version/10_...,NaN,Christlichdemokratische Volkspartei der Schweiz
9,https://politics.ld.admin.ch/party-version/10_...,https://politics.ld.admin.ch/party-version/10_...,Konservativ-Christlichsoziale Volkspartei,NaN


In [15]:
from ipycytoscape import CytoscapeWidget
import networkx as nx

my_style = [
    {
        "selector": "node",
        "style": {
            "label": "data(name)",
            "background-color": "#0074D9",
            "color": "red",
            "text-valign": "center",
            "text-halign": "center",
            "font-size": 10,
            "width": 20,
            "height": 20,
        },
    },
    {
        "selector": "edge.directed",
        "style": {
            "line-color": "#0074D9",
            "target-arrow-color": "#0074D9",
            "target-arrow-shape": "triangle",
            "curve-style": "bezier",
        },
    },
]

ids = pd.Series.tolist(df['source']) + pd.Series.tolist(df['target'])
names = pd.Series.tolist(df['source_name']) + pd.Series.tolist(df['target_name'])

node_df = pd.DataFrame({'id': ids, 'name': names})
node_df = node_df.drop_duplicates(subset=['id']).reset_index(drop=True)

G = nx.from_pandas_edgelist(df, source='source', target='target', create_using=nx.DiGraph())
nx.set_node_attributes(G, node_df.set_index('id').to_dict('index'))

cyto = CytoscapeWidget()
cyto.graph.add_graph_from_networkx(G, directed=True)
cyto.set_style(my_style)
cyto.set_layout(name = "dagre", rankDir = "LR")

cyto

/Users/bene/Nextcloud BFH/Projekte/26_BK_Parties/rdf-parties/.venv/lib/python3.13/site-packages/jupyter_client/session.py:727: UserWarning: Message serialization failed with:
Out of range float values are not JSON compliant: nan
Supporting this message is deprecated in jupyter-client 7, please make sure your message is JSON-compliant
  content = self.pack(content)


CytoscapeWidget(cytoscape_layout={'name': 'dagre', 'rankDir': 'LR'}, cytoscape_style=[{'selector': 'node', 'st…